# Data Ingestion & Cleaning

A reusable pandas workflow for getting raw data into a clean, analysis-ready state.

**Pipeline:** load (CSV / JSON / text) → inspect → normalize column names → handle missing values → EDA → export cleaned CSV.

The notebook generates a small messy sample dataset so it runs end to end with no external files. Point `load_data` at your own file to use it for real.

## 1. Imports

In [1]:
import json
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## 2. Create a sample dataset

We write a deliberately messy CSV (inconsistent column names, missing values), plus a JSON and a text file, to demonstrate the multi-format loader.

In [2]:
DATA_DIR = Path("sample_data")
DATA_DIR.mkdir(exist_ok=True)

csv_text = """Customer Name,  Email ,Age,Country,Order Amount ($),Product Category,Membership,Signup Date
Alice Johnson,alice@example.com,34,USA,120.50,Electronics,Gold,2023-01-15
Bob Smith,,29,UK,,Books,Silver,2023-02-20
Carol White,carol@example.com,,USA,89.99,Electronics,Gold,2023-03-01
David Brown,david@example.com,41,Canada,45.00,Home,Bronze,
Eve Davis,eve@example.com,42,USA,230.75,Electronics,Gold,2023-01-25
Frank Miller,frank@example.com,37,UK,67.20,Books,,2023-04-12
Grace Lee,grace@example.com,28,Canada,,Home,Silver,2023-02-08
Henry Wilson,,55,USA,310.00,Electronics,Gold,2023-05-19
Ivy Chen,ivy@example.com,33,UK,99.99,Books,Bronze,2023-03-22
Jack Taylor,jack@example.com,,Canada,150.45,Home,Silver,
Karen Moore,karen@example.com,46,USA,72.30,Books,Gold,2023-06-01
Leo Martin,leo@example.com,31,UK,205.10,Electronics,,2023-04-30
Mia Clark,,38,USA,,Home,Silver,2023-05-15
Noah Lewis,noah@example.com,49,Canada,128.80,Electronics,Bronze,2023-02-14
Olivia Hall,olivia@example.com,27,USA,54.60,Books,Gold,2023-06-20
"""
(DATA_DIR / "orders.csv").write_text(csv_text, encoding="utf-8")

reviews = [
    {"review_id": 1, "Product Name": "Wireless Mouse", "Rating": 4, "Verified Buyer": True, "Review Text": "Great value for the price."},
    {"review_id": 2, "Product Name": "USB-C Cable", "Rating": None, "Verified Buyer": False, "Review Text": "Stopped working after a week."},
    {"review_id": 3, "Product Name": "Mechanical Keyboard", "Rating": 5, "Verified Buyer": True, "Review Text": "Fantastic tactile feel."},
    {"review_id": 4, "Product Name": "Laptop Stand", "Rating": 2, "Verified Buyer": True, "Review Text": None},
]
(DATA_DIR / "reviews.json").write_text(json.dumps(reviews, indent=2), encoding="utf-8")

notes = "First quarter looked strong.\n\nElectronics outperformed every other category.\n   \nMembership churn stayed flat.\n"
(DATA_DIR / "notes.txt").write_text(notes, encoding="utf-8")

sorted(p.name for p in DATA_DIR.iterdir())

['notes.txt', 'orders.csv', 'reviews.json']

## 3. Load data

One entry point that dispatches on file extension. CSV and JSON map straight to DataFrames; text becomes a single `text` column with blank lines dropped.

In [3]:
def load_data(path) -> pd.DataFrame:
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix == ".json":
        return pd.read_json(path)
    if suffix in {".txt", ".text"}:
        lines = path.read_text(encoding="utf-8").splitlines()
        return pd.DataFrame({"text": [ln.strip() for ln in lines if ln.strip()]})
    raise ValueError(f"Unsupported file type: {suffix}")


df = load_data(DATA_DIR / "orders.csv")
df

,Customer Name,Email,Age,Country,Order Amount ($),Product Category,Membership,Signup Date
0,Alice Johnson,alice@example.com,34.0,USA,120.50,Electronics,Gold,2023-01-15
1,Bob Smith,NaN,29.0,UK,NaN,Books,Silver,2023-02-20
2,Carol White,carol@example.com,NaN,USA,89.99,Electronics,Gold,2023-03-01
3,David Brown,david@example.com,41.0,Canada,45.00,Home,Bronze,NaN
4,Eve Davis,eve@example.com,42.0,USA,230.75,Electronics,Gold,2023-01-25
5,Frank Miller,frank@example.com,37.0,UK,67.20,Books,NaN,2023-04-12
6,Grace Lee,grace@example.com,28.0,Canada,NaN,Home,Silver,2023-02-08
7,Henry Wilson,NaN,55.0,USA,310.00,Electronics,Gold,2023-05-19
8,Ivy Chen,ivy@example.com,33.0,UK,99.99,Books,Bronze,2023-03-22
9,Jack Taylor,jack@example.com,NaN,Canada,150.45,Home,Silver,NaN


## 4. Inspect the raw data

In [4]:
print("shape:", df.shape)
df.dtypes

shape: (15, 8)


Customer Name           str
  Email                 str
Age                 float64
Country                 str
Order Amount ($)    float64
Product Category        str
Membership              str
Signup Date             str
dtype: object

In [5]:
df.head()

,Customer Name,Email,Age,Country,Order Amount ($),Product Category,Membership,Signup Date
0,Alice Johnson,alice@example.com,34.0,USA,120.50,Electronics,Gold,2023-01-15
1,Bob Smith,NaN,29.0,UK,NaN,Books,Silver,2023-02-20
2,Carol White,carol@example.com,NaN,USA,89.99,Electronics,Gold,2023-03-01
3,David Brown,david@example.com,41.0,Canada,45.00,Home,Bronze,NaN
4,Eve Davis,eve@example.com,42.0,USA,230.75,Electronics,Gold,2023-01-25


In [6]:
df.isna().sum()

Customer Name       0
  Email             3
Age                 2
Country             0
Order Amount ($)    3
Product Category    0
Membership          2
Signup Date         2
dtype: int64

## 5. Normalize column names

Lowercase, strip surrounding whitespace, drop punctuation, and collapse spaces into underscores so columns are consistent and easy to reference.

In [7]:
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.str.strip()
        .str.lower()
        .str.replace(r"[^\w\s]", "", regex=True)
        .str.replace(r"\s+", "_", regex=True)
    )
    return df


df = normalize_columns(df)
list(df.columns)

['customer_name',
 'email',
 'age',
 'country',
 'order_amount_',
 'product_category',
 'membership',
 'signup_date']

## 6. Handle missing values

Drop fully empty rows and columns, then fill what remains: numeric columns with the median, and text / categorical columns with an explicit `"unknown"` placeholder so gaps stay visible instead of being replaced by a guessed real value.

In [8]:
def handle_missing(df: pd.DataFrame) -> pd.DataFrame:
    df = df.dropna(how="all").dropna(axis=1, how="all").copy()
    for col in df.columns:
        if not df[col].isna().any():
            continue
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna("unknown")
    return df


before = df.isna().sum().sum()
df = handle_missing(df)
after = df.isna().sum().sum()
print(f"missing cells: {before} -> {after}")
df.isna().sum()

missing cells: 12 -> 0


customer_name       0
email               0
age                 0
country             0
order_amount_       0
product_category    0
membership          0
signup_date         0
dtype: int64

## 7. Exploratory data analysis

In [9]:
df.head()

,customer_name,email,age,country,order_amount_,product_category,membership,signup_date
0,Alice Johnson,alice@example.com,34.0,USA,120.500,Electronics,Gold,2023-01-15
1,Bob Smith,unknown,29.0,UK,110.245,Books,Silver,2023-02-20
2,Carol White,carol@example.com,37.0,USA,89.990,Electronics,Gold,2023-03-01
3,David Brown,david@example.com,41.0,Canada,45.000,Home,Bronze,unknown
4,Eve Davis,eve@example.com,42.0,USA,230.750,Electronics,Gold,2023-01-25


In [10]:
df.describe()

,age,order_amount_
count,15.000000,15.000000
mean,37.600000,127.027667
std,7.980333,71.924273
min,27.000000,45.000000
25%,32.000000,81.145000
50%,37.000000,110.245000
75%,41.500000,139.625000
max,55.000000,310.000000


In [11]:
df.describe(include="object")

/tmp/ipykernel_601/702825166.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include="object")


,customer_name,email,country,product_category,membership,signup_date
count,15,15,15,15,15,15
unique,15,13,3,3,4,14
top,Alice Johnson,unknown,USA,Electronics,Gold,unknown
freq,1,3,7,6,6,2


In [12]:
df["country"].value_counts()

country
USA       7
UK        4
Canada    4
Name: count, dtype: int64

In [13]:
df["product_category"].value_counts()

product_category
Electronics    6
Books          5
Home           4
Name: count, dtype: int64

In [14]:
df["membership"].value_counts(normalize=True).round(3)

membership
Gold       0.400
Silver     0.267
Bronze     0.200
unknown    0.133
Name: proportion, dtype: float64

## 8. Export the cleaned CSV

In [15]:
OUTPUT_PATH = Path("cleaned_orders.csv")
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(df)} rows -> {OUTPUT_PATH}")

Saved 15 rows -> cleaned_orders.csv


## 9. Same loader, other formats

The JSON and text files flow through the exact same `load_data` entry point.

In [16]:
reviews_df = normalize_columns(load_data(DATA_DIR / "reviews.json"))
reviews_df

,review_id,product_name,rating,verified_buyer,review_text
0,1,Wireless Mouse,4.0,True,Great value for the price.
1,2,USB-C Cable,NaN,False,Stopped working after a week.
2,3,Mechanical Keyboard,5.0,True,Fantastic tactile feel.
3,4,Laptop Stand,2.0,True,NaN


In [17]:
notes_df = load_data(DATA_DIR / "notes.txt")
notes_df

,text
0,First quarter looked strong.
1,Electronics outperformed every other category.
2,Membership churn stayed flat.
